# 04 - Repeated-run stability (GPT-5.5)

Three independent runs of the Baseline and Full Framework conditions on a
stratified 48-sample subset, quantifying nondeterminism.

Run cells in order. Run 1 already exists in `frontier_gpt55.csv`; this notebook
adds runs 2 and 3 and merges all three.

In [1]:
!pip -q install openai
from google.colab import drive; drive.mount('/content/drive')

import sys
from pathlib import Path
import pandas as pd

ROOT = Path('/content/drive/MyDrive/LLM_Security_Paper')
WORK = ROOT / 'revision_2026'
sys.path.insert(0, str(WORK))
import vulnbench as vb

BENCH = pd.read_csv(WORK / 'benchmark_360_metadata.csv')
print(len(BENCH), 'samples |', BENCH.true_label.value_counts().to_dict())

Mounted at /content/drive
360 samples | {'Safe': 180, 'Vulnerable': 180}


In [2]:
import os, getpass
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')
from openai import OpenAI
client = OpenAI()

OpenAI API key: ··········


In [3]:
# --- stability subset: 3 CWE x 2 labels x 8 = 48 samples, fully symmetric ---
STAB = (BENCH.groupby(['cwe', 'true_label'], group_keys=False)
             .apply(lambda d: d.sample(8, random_state=42)))
STAB = STAB.sort_values('sample_id').reset_index(drop=True)

print(len(STAB), 'samples')
print(pd.crosstab(STAB.cwe, STAB.true_label))
print('\ncalls to make:', len(STAB) * 2 * 2, '(48 samples x 2 conditions x 2 extra runs)')
print('estimated cost: ~$2.9')

48 samples
true_label  Safe  Vulnerable
cwe                         
CWE_78         8           8
CWE_89         8           8
CWE_98         8           8

calls to make: 192 (48 samples x 2 conditions x 2 extra runs)
estimated cost: ~$2.9


/tmp/ipykernel_2379/729484242.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: d.sample(8, random_state=42)))


In [4]:
# --- runs 2 and 3 (run 1 is already in frontier_gpt55.csv) ---
res_stab = vb.run_experiment(
    bench        = STAB,
    dataset_root = ROOT,
    conditions   = [('A', 'clean'), ('E', 'clean')],
    model        = 'gpt-5.5-2026-04-23',
    temperature  = None,
    run_ids      = (2, 3),
    out_csv      = WORK / 'stability_gpt55.csv',
    client       = client,
)

192 calls to make with gpt-5.5-2026-04-23
  50/192  |  tokens in/out 11298/25563  |  running cost $0.823
  100/192  |  tokens in/out 27171/43311  |  running cost $1.435
  150/192  |  tokens in/out 38985/65289  |  running cost $2.154
  192/192  |  tokens in/out 53012/81443  |  running cost $2.708

done — 192 rows in /content/drive/MyDrive/LLM_Security_Paper/revision_2026/stability_gpt55.csv
this session cost ≈ $2.708


In [5]:
# --- merge the three runs and quantify prediction stability ---
run1 = pd.read_csv(WORK / 'frontier_gpt55.csv')
run1 = run1[run1.sample_id.isin(STAB.sample_id) &
            run1.condition.isin(['A_clean', 'E_clean'])].copy()
run1['run_id'] = 1

allruns = pd.concat([run1, pd.read_csv(WORK / 'stability_gpt55.csv')],
                    ignore_index=True)
print('rows:', len(allruns), '| runs:', sorted(allruns.run_id.unique()))

# per (sample, condition): how many distinct predictions across the 3 runs?
g = (allruns.groupby(['sample_id', 'condition'])
             .agg(n_runs=('prediction', 'size'),
                  n_distinct=('prediction', 'nunique'))
             .reset_index())
g['unstable'] = g.n_distinct > 1

print('\n--- prediction stability across 3 runs ---')
for c, d in g.groupby('condition'):
    k, n = int(d.unstable.sum()), len(d)
    print(f'  {c}: {k}/{n} unstable ({k/n:.1%})')
k, n = int(g.unstable.sum()), len(g)
print(f'  overall: {k}/{n} unstable ({k/n:.1%})')

# accuracy per run, to show the effect direction is stable
print('\n--- accuracy per run ---')
acc = (allruns.assign(correct=lambda d: d.prediction == d.true_label)
              .groupby(['condition', 'run_id']).correct.mean().unstack().round(3))
print(acc.to_string())
print('\nA - E gap per run:', (acc.loc['A_clean'] - acc.loc['E_clean']).round(3).to_dict())

allruns.to_csv(WORK / 'stability_merged_3runs.csv', index=False)
print('\nmerged file written: stability_merged_3runs.csv')

rows: 288 | runs: [np.int64(1), np.int64(2), np.int64(3)]

--- prediction stability across 3 runs ---
  A_clean: 6/48 unstable (12.5%)
  E_clean: 4/48 unstable (8.3%)
  overall: 10/96 unstable (10.4%)

--- accuracy per run ---
run_id         1      2      3
condition                     
A_clean    0.771  0.729  0.750
E_clean    0.729  0.771  0.792

A - E gap per run: {1: 0.042, 2: -0.042, 3: -0.042}

merged file written: stability_merged_3runs.csv
